<a href="https://colab.research.google.com/github/RomanPhys/External/blob/main/external_context_v6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Внешний контекст, версия v6

Прошлый прогон показал две вещи. На SASRec авторов агрегации состояний из статьи при честном протоколе ничего не дают, а память переходов даёт небольшой, но устойчивый прирост. И ещё важнее, разбиение по пользователям в принципе не оставляет внешнему контексту шанса. Энкодер там учится на всей истории обучающих пользователей, включая время после запроса тестового, то есть уже видел, что тогда происходило в мире.

Этот ноутбук проверяет другой подход в постановке, где у контекста шанс есть. Энкодер учится только на прошлом, до момента T_v, и дальше заморожен, а проверка идёт на событиях после T_v, как у модели в проде. Внешний контекст собирается не из состояний других пользователей, а из того, что они сделали. Это память переходов, то есть что делали дальше другие в похожей ситуации, и инновации, то есть что другие сделали за последние дни сверх того, что им предсказывала модель. Проверяются два канала. Первый про то, что человек сделает (следующий фильм или MCC), второй про то, когда (будет ли он активен в ближайшую неделю).

GPU не нужен, всё считается на процессоре. MovieLens занимает около 15 минут, Churn около 20 минут. Ещё примерно полчаса уйдёт на Churn в старой постановке с разбиением по клиентам, для сравнения. Запусти все ячейки через Run all.

## Настройки

USE_DRIVE лучше оставить True. Тогда всё сохраняется на Google Drive в папку ext_work_v6, и после обрыва сессии посчитанное пропустится. BANK задаёт датасеты авторов, по умолчанию только churn. RUN_BANK_USERSPLIT включает прогон Churn в старой постановке, как часть B в v5.

In [1]:
USE_DRIVE = True
BANK = ["churn"]
BANK_MAX_USERS = 12000
RUN_BANK_USERSPLIT = True

import os, json, glob, zipfile, shutil, subprocess, time
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/ext_work_v6"
else:
    WORK = "/content/work"
os.makedirs(WORK, exist_ok=True)
print("результаты будут в", WORK)

Mounted at /content/drive
результаты будут в /content/drive/MyDrive/ext_work_v6


## 1. Код

Когда появится кнопка выбора файла, загрузи external_context_v6.zip. Если архив уже лежит в папке ext_work_v6 на Drive, ноутбук возьмёт его оттуда.

In [2]:
CODE = "/content/ext"
if not os.path.exists(f"{CODE}/code/run_temporal.py"):
    shutil.rmtree(CODE, ignore_errors=True)
    found = sorted(glob.glob(f"{WORK}/external_context_v*.zip"))
    if found:
        path = found[-1]
    else:
        from google.colab import files
        path = next(iter(files.upload()))
        if USE_DRIVE:
            shutil.copy(path, WORK)
    zipfile.ZipFile(path).extractall(CODE)
assert os.path.exists(f"{CODE}/code/run_temporal.py"), "это старый архив, нужен external_context_v6.zip"
print("код на месте", sorted(os.listdir(CODE)))


def run(script, W, log, **env):
    # каждый шаг в отдельном процессе с чистыми переменными EXT_
    e = {k: v for k, v in os.environ.items() if not k.startswith("EXT_")}
    e.update({k: str(v) for k, v in env.items()})
    e["EXT_WORK"] = W
    t0 = time.time()
    with open(f"{W}/{log}", "w") as f:
        p = subprocess.run(["python", script], env=e, stdout=f, stderr=subprocess.STDOUT, cwd=f"{CODE}/code")
    if p.returncode != 0:
        print(open(f"{W}/{log}").read()[-3000:])
        raise RuntimeError(f"{script} упал, хвост лога выше")
    print(f"    {script} готов за {time.time() - t0:.0f} с")


def show(script, W, *args, **env):
    e = {k: v for k, v in os.environ.items() if not k.startswith("EXT_")}
    e.update({k: str(v) for k, v in env.items()})
    e["EXT_WORK"] = W
    p = subprocess.run(["python", script, *args], env=e, capture_output=True, text=True, cwd=f"{CODE}/code")
    print(p.stdout)
    if p.returncode != 0:
        print(p.stderr[-3000:])

Saving external_context_v6.zip to external_context_v6.zip
код на месте ['README.md', 'code', 'colab', 'results', 'torch_patch']


## 2. Данные

MovieLens-1M скачивается с сайта GroupLens. Churn берётся из папки preprocessed на Google Drive авторов (ссылка есть в README их репозитория). В v5 скачивание упало из-за моей ошибки, я передавал в gdown параметр, которого нет в новой версии. Теперь ноутбук сам смотрит, какие параметры поддерживает установленный gdown.

Если скачать всё равно не выйдет, ячейка напечатает, что видит в папке. Тогда открой https://drive.google.com/drive/folders/1VFGL6ZOtnP7ErPyVlmQv62NJTXUs8ekY, скачай из preprocessed файл churn.parquet, положи его в ext_work_v6/bank_data на своём Drive и запусти ячейку снова.

In [3]:
!test -f /content/ml-1m/ratings.dat || (wget -q https://files.grouplens.org/datasets/movielens/ml-1m.zip -O /content/ml-1m.zip && unzip -o -q /content/ml-1m.zip -d /content/)
!wc -l /content/ml-1m/ratings.dat

BANK_DIR = f"{WORK}/bank_data"
os.makedirs(BANK_DIR, exist_ok=True)
for n in BANK:                                   # файл, положенный вручную в прошлые папки
    for cand in glob.glob(f"{os.path.dirname(WORK)}/ext_work_v*/bank_data/{n}.parquet") + glob.glob(f"/content/{n}.parquet"):
        if not os.path.exists(f"{BANK_DIR}/{n}.parquet"):
            shutil.copy(cand, f"{BANK_DIR}/{n}.parquet")
need = [n for n in BANK if not os.path.exists(f"{BANK_DIR}/{n}.parquet")]
if need:
    !pip -q install -U gdown
    import inspect, gdown
    params = inspect.signature(gdown.download_folder).parameters
    if "skip_download" not in params:
        print("эта версия gdown не умеет получать список файлов без скачивания, скачай файл вручную")
    else:
        kw = dict(id="1VFGL6ZOtnP7ErPyVlmQv62NJTXUs8ekY", skip_download=True, quiet=True)
        if "remaining_ok" in params:
            kw["remaining_ok"] = True
        try:
            listing = gdown.download_folder(**kw) or []
            paths = [(getattr(f, "id", None), getattr(f, "path", "") or getattr(f, "local_path", "")) for f in listing]
            for n in need:
                hits = [(i, p) for i, p in paths if os.path.basename(p) == f"{n}.parquet"]
                hits.sort(key=lambda x: "preprocessed" not in x[1])
                if hits:
                    gdown.download(id=hits[0][0], output=f"{BANK_DIR}/{n}.parquet", quiet=False)
                else:
                    print(f"файла {n}.parquet в папке не видно, вот что там есть")
                    for i, p in paths[:60]:
                        print("   ", p)
        except Exception as err:
            print("автоматически скачать не вышло", repr(err)[:300])
import pandas as pd
for n in BANK:
    p = f"{BANK_DIR}/{n}.parquet"
    if os.path.exists(p):
        df = pd.read_parquet(p, columns=["user_id", "mcc_code", "timestamp"])
        print(n, len(df), "транзакций,", df.user_id.nunique(), "клиентов,", df.mcc_code.nunique(), "MCC,",
              "время от", df.timestamp.min(), "до", df.timestamp.max())
    else:
        print(f"нет файла {p}, скачай его вручную по инструкции выше")

1000209 /content/ml-1m/ratings.dat


Downloading...
From: https://drive.google.com/uc?id=1c-aF7u5M6MotDRlljJSgwQmVHVS4abfX
To: /content/drive/MyDrive/ext_work_v6/bank_data/churn.parquet
100%|██████████| 3.62M/3.62M [00:00<00:00, 4.23MB/s]


churn 490513 транзакций, 5000 клиентов, 344 MCC, время от 2016-10-07 00:00:00 до 2018-04-02 23:20:35


## 3. Временной сплит

Для каждого датасета скрипт run_temporal.py учит энкодер на событиях до T_v (80% всех событий по времени), валидацию берёт из следующих 10% и тест из последних 10%. Потом он сравнивает источники внешнего контекста в каналах что и когда и считает критерии, которые заранее говорят, может ли контекст помочь. Окна для MovieLens 30, 90 и 365 дней, потому что после T_v событий там мало, для банка 7, 30 и 90 дней.

In [4]:
jobs = [("ml1m", dict(EXT_ML1M_DAT="/content/ml-1m/ratings.dat", EXT_WINDOWS="30,90,365"))]
for n in BANK:
    src = f"{BANK_DIR}/{n}.parquet"
    if os.path.exists(src):
        jobs.append((n, dict(EXT_DATASET="bank", EXT_BANK_FILE=src, EXT_BANK_MAX_USERS=BANK_MAX_USERS, EXT_WINDOWS="7,30,90")))
for name, env in jobs:
    W = f"{WORK}/temporal_{name}"
    os.makedirs(W, exist_ok=True)
    print(name)
    if not os.path.exists(f"{W}/results_temporal.json"):
        run("run_temporal.py", W, "temporal.stdout", **env)
    show("summarize.py", W)

ml1m
    run_temporal.py готов за 524 с
Результаты из папки /content/drive/MyDrive/ext_work_v6/temporal_ml1m

Временной сплит, энкодер обучен только на событиях до T_v и заморожен
  до T_v 80% событий, дней на валидацию 26, на тест 794
  канал что, база NDCG@10 0.0716, запросов в тесте 5389 у 1163 пользователей
    сжатие g = 0                                 +0.0001 [-0.0004; +0.0006]
    G-состояния, вектор как в статье             +0.0004 [-0.0004; +0.0013]
    G-популярность за окно                       +0.0003 [-0.0001; +0.0007]
    G-инновации                                  +0.0012 [+0.0001; +0.0023]
    L-память, вся                                +0.0087 [+0.0052; +0.0121]
    L-память, старая                             +0.0083 [+0.0055; +0.0110]
    L-память, свежая                             +0.0064 [+0.0040; +0.0087]
    L-память + G-инновации                       +0.0087 [+0.0052; +0.0121]
    сравнения
      G-инновации против G-состояний             +0.0007 [-0.0005

## 4. Churn в старой постановке

Это та же часть B, что в v5, с разбиением по клиентам. Она нужна для сравнения, чтобы было видно, как одни и те же источники контекста ведут себя при разбиении по клиентам и при временном сплите. Кеши этой части занимают на Drive около гигабайта, после отправки результатов файлы .pkl и .npz можно удалить.

In [5]:
if RUN_BANK_USERSPLIT:
    for n in BANK:
        src = f"{BANK_DIR}/{n}.parquet"
        if not os.path.exists(src):
            print(f"пропускаю {n}, нет данных")
            continue
        W = f"{WORK}/bank_{n}_split0"
        os.makedirs(W, exist_ok=True)
        env = dict(EXT_DATASET="bank", EXT_BANK_FILE=src, EXT_BANK_MAX_USERS=BANK_MAX_USERS,
                   EXT_SPLIT_SEED=0, EXT_STRICT_PAST=1, EXT_WINDOWS="7,30,90")
        print(n)
        for script, out, log in (("run_encoder2.py", "encoder_msema.npz", "encoder_train.log"),
                                 ("run_context.py", "cache_context.pkl", "context.log"),
                                 ("run_experiments.py", "results.json", "experiments.stdout"),
                                 ("run_shrink_control.py", "results_shrink_control.json", "shrink.log"),
                                 ("run_diagnostics.py", "diagnostics.json", "diagnostics.log"),
                                 ("run_retrieval.py", "results_retrieval.json", "retrieval.stdout"),
                                 ("run_time_local.py", "results_time_local.json", "time_local.stdout")):
            if not os.path.exists(f"{W}/{out}"):
                run(script, W, log, **env)

churn
    run_encoder2.py готов за 46 с
    run_context.py готов за 60 с
    run_experiments.py готов за 54 с
    run_shrink_control.py готов за 2 с
    run_diagnostics.py готов за 1 с
    run_retrieval.py готов за 208 с
    run_time_local.py готов за 118 с


## 5. Сводка и сохранение

Скопируй весь вывод следующей ячейки и пришли мне вместе с архивом из последней ячейки.

In [6]:
for W in sorted(glob.glob(f"{WORK}/temporal_*")) + sorted(glob.glob(f"{WORK}/bank_*_split0")):
    if os.path.exists(f"{W}/results_temporal.json") or os.path.exists(f"{W}/results.json"):
        print("=" * 30, os.path.basename(W), "=" * 30)
        if os.path.exists(f"{W}/encoder_log.json"):
            print(json.load(open(f"{W}/encoder_log.json"))["info"])
        show("summarize.py", W)

============================== temporal_churn ==============================
Результаты из папки /content/drive/MyDrive/ext_work_v6/temporal_churn

Временной сплит, энкодер обучен только на событиях до T_v и заморожен
  до T_v 80% событий, дней на валидацию 36, на тест 206
  канал что, база NDCG@10 0.5315, запросов в тесте 4601 у 981 пользователей
    сжатие g = 0                                 +0.0000 [+0.0000; +0.0000]
    G-состояния, вектор как в статье             +0.0000 [+0.0000; +0.0000]
    G-популярность за окно                       +0.0000 [+0.0000; +0.0000]
    G-инновации                                  +0.0012 [-0.0001; +0.0025]
    L-память, вся                                +0.0046 [+0.0020; +0.0073]
    L-память, старая                             +0.0042 [+0.0019; +0.0065]
    L-память, свежая                             +0.0015 [-0.0004; +0.0033]
    L-память + G-инновации                       +0.0059 [+0.0032; +0.0088]
    сравнения
      G-инновации против G-с

In [7]:
!cd {WORK} && rm -f /content/results_v6.zip && zip -q -r /content/results_v6.zip . -x "*.npz" "*.pkl" "*.pth" "*.zip" "*.parquet"
if USE_DRIVE:
    shutil.copy("/content/results_v6.zip", WORK)
from google.colab import files
files.download("/content/results_v6.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Как читать результаты

В канале что главные строки это G-инновации против G-состояний и свежая память против старой. Если инновации обгоняют среднее состояний, значит контекст в целом надо брать не из того, кто такие другие люди, а из того, насколько они удивили модель. Если свежая память обгоняет старую, значит внешний контекст приносит то, чего модель не видела при обучении, и это ровно тезис статьи о меняющемся мире.

В канале когда смотри, помогают ли инновации активности. Гипотеза такая, что общий для всех сдвиг сильнее всего сказывается на том, когда люди действуют, а похожие ситуации на том, что именно они делают. Критерии до обучения должны с этим согласовываться. Отношение доли остатка, объяснённой неделей, к той же доле при перемешивании заметно больше единицы там, где контекст в целом помогает.

Если что-то пошло не так, ячейка покажет хвост лога упавшего шага, его и пришли.